# CENG313 Introduction to Data Science — Assignment 1: EDA & Baseline Models
## Competition: [CAFA 6 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-6-protein-function-prediction)

This notebook carries out an Exploratory Data Analysis (EDA) and implements Baseline Models for the CAFA 6 Protein Function Prediction challenge.

All the reusable code is under the `src/` folder.

## 1. Environment Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path so `src` can be imported cleanly
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    resolve_data_paths,
    parse_fasta,
    load_train_terms,
    load_taxonomy,
    load_ia_weights,
    parse_go_obo_summary,
    inspect_non_canonical_amino_acids
)

plt.rcParams["figure.dpi"] = 110

## 2. Dataset Configuration
Set `DATA_DIR` to point to the directory containing the competition files.

In [ ]:
DATA_DIR = Path("data/cafa-6-protein-function-prediction")
PATHS = resolve_data_paths(DATA_DIR)

for name, path in PATHS.items():
    print(f"{name:<18}: {path} (exists={path.exists()})")

In [ ]:
# Load core dataset files
train_seq_df = parse_fasta(PATHS["train_fasta"])
test_seq_df = parse_fasta(PATHS["test_fasta"])
terms_df = load_train_terms(PATHS["train_terms"])

tax_df = load_taxonomy(PATHS["train_taxonomy"]) if PATHS["train_taxonomy"].exists() else None
test_tax_df = load_taxonomy(PATHS["test_taxonomy"]) if PATHS["test_taxonomy"].exists() else None
ia_df = load_ia_weights(PATHS["ia"]) if PATHS["ia"].exists() else None
obo_summary = parse_go_obo_summary(PATHS["go_obo"])

print(f"Training sequences : {len(train_seq_df):,}")
print(f"Test sequences     : {len(test_seq_df):,}")
print(f"Total annotations  : {len(terms_df):,}")
print(f"Unique GO terms    : {terms_df['term'].nunique():,}")
print(f"GO OBO summary     : {obo_summary}")

## 3. Exploratory Data Analysis

### 3.1 Data Integrity and Missing/Corrupted Value Check
We verify whether there are any missing (`NaN`) records, inspect sequence length distributions, and scan all protein sequences for non-canonical/ambiguous IUPAC residues (`B`, `U`, `X`, `Z`, `O`, `J`) or corrupted/invalid characters outside the 20 standard amino acids (`ACDEFGHIKLMNPQRSTVWY`).

In [ ]:
print("---- Missing Values in train_seq_df ")
print(train_seq_df.isna().sum())
print("\n---- Missing Values in terms_df ")
print(terms_df.isna().sum())

print("\n---- Sequence Length Summary Statistics (Train vs Test) ")
summary_stats = pd.DataFrame({
    "Train_Length": train_seq_df["length"].describe(),
    "Test_Length": test_seq_df["length"].describe(),
})
display(summary_stats)

print("\n--- Non-Canonical / Invalid Amino Acid Overview ---")
aa_overview_df, aa_residue_df = inspect_non_canonical_amino_acids({
    "Train": train_seq_df,
    "Test": test_seq_df,
})
display(aa_overview_df)

print("\n--- Detailed Breakdown of Non-Canonical Amino Acid Residues ---")
display(aa_residue_df)

### 3.2 Sub-Ontology (Aspect) Breakdown & Top GO Terms
Gene Ontology terms are organized into three independent sub-ontologies:
- **BPO**: Biological Process
- **CCO**: Cellular Component
- **MFO**: Molecular Function

In [ ]:
fig1 = plot_aspect_and_top_terms(terms_df, top_n=20)
plt.show()